**Universidad Internacional de La Rioja (UNIR) - Máster Universitario en Inteligencia Artificial - Procesamiento del Lenguaje Natural** 

***
Datos del alumno (Nombre y Apellidos):

Fecha:
***

<span style="font-size: 20pt; font-weight: bold; color: #0098cd;">Trabajo: Caracterización de textos</span>

**Objetivos** 

Con esta actividad se tratará de que el alumno se familiarice con el manejo de la librería spacy, así como con los conceptos básicos de manejo de las técnicas NER

**Descripción**

En esta actividad debes procesar de forma automática un texto en lenguaje natural para detectar características básicas en el mismo, y para identificar y etiquetar las ocurrencias de conceptos como localización, moneda, empresas, etc.

En la primera parte del ejercicio se proporciona un código fuente a través del cual se lee un archivo de texto y se realiza un preprocesado del mismo. En esta parte el alumno tan sólo debe ejecutar y entender el código proporcionado.

En la segunda parte del ejercicio se plantean una serie de preguntas que deben ser respondidas por el alumno. Cada pregunta deberá responderse con un fragmento de código fuente que esté acompañado de la explicación correspondiente. Para elaborar el código solicitado, el alumno deberá visitar la documentación de la librería spacy, cuyos enlaces se proporcionarán donde corresponda.

# Parte 1: carga y preprocesamiento del texto a analizar

Observa las diferentes librerías que se están importando.

In [30]:
import pathlib
import spacy
import pandas as pd
from spacy import displacy
import csv
import es_core_news_md

El siguiente código simplemente carga y preprocesa el texto. Para ello, lo primero que hace es cargar un modelo de lenguaje previamente entrenado. En este caso, se utiliza <i>es_core_news_md</i>: 

https://spacy.io/models/es#es_core_news_md


In [31]:
nlp = es_core_news_md.load()

El objeto <i>nlp</i> permite utilizar el modelo de lenguaje cargado, de forma que se puede procesar un texto y obtenerlo en su versión preprocesada. Así, nos permite realizar las diferentes tareas. En este caso, vamos a utilizar el pipeline para hacer un preprocesamiento básico, que consiste en tokenizar el texto.

In [32]:
filename = "./02Dataset_anonimizado.csv"
lines_number = 10000
data = pd.read_csv(filename,delimiter=';', encoding='utf-8', low_memory=False, encoding_errors="ignore")

El código anterior carga el archivo CSV (opcionalmente con un límite de líneas a leer) y genera la variable <i>data</i>, que contiene un Dataframe (https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) con los datos leídos del CSV.

Te vendrá bien conocer la siguiente documentación:
<ul>
    <li>https://spacy.io/api/doc</li>
    <li>https://spacy.io/api/token</li>
    <li>https://spacy.io/api/morphology#morphanalysis</li>
</ul>

### Playground

Utiliza este espacio para hacer pruebas y ensayos con las variables generadas con el código previo. A modo de ejemplo, se ofrece código que realiza las siguientes tareas: 


- leer un número dado de líneas del Dataframe y generar dos listas con los valores (se pueden leer directamente del DataFrame, se muestra el ejemplo como una opción más)
- procesar el texto de cada comentario


Para procesarlo, hay utilizar el objeto <i>nlp</i> y así obtener objetos de la clase <i>Doc</i> (https://spacy.io/api/doc)

Visita la documentación de dicha clase y experimenta probando las diferentes funciones y atributos 

In [117]:
lines_number = min(lines_number, len(data))

# Inicializar contadores generales
total_odio = 0
total_no_odio = 0
total_sin_clasificar = 0
total_palabras = 0
palabras_odio = 0
palabras_no_odio = 0
no_odio_con_ner = 0
odio_con_ner = 0

# Contadores de signos de exclamación
exclamaciones_odio = 0
exclamaciones_no_odio = 0

# Acumuladores de riqueza léxica
riqueza_lexica_odio = 0
riqueza_lexica_no_odio = 0

# Contadores de pronombres y palabras por grupo
pronombres_odio = 0
pronombres_no_odio = 0
palabras_pronombres_odio = 0
palabras_pronombres_no_odio = 0

# Definir algunas palabras para detectar coincidencias de odio
palabras_diccionario_odio = {
    "idiota", "estúpido", "imbécil", "inútil",
    "basura", "ignorante", "asco", "detestable"
}

# Inicializar contadores de frecuencia del diccionario
frecuencia_terminos_odio = 0
frecuencia_terminos_no_odio = 0
comentarios_con_terminos_odio = 0
comentarios_con_terminos_no_odio = 0
frecuencia_por_termino = {}

# Diccionario para corregir errores de codificación de caracteres
replacements = {
    "Ã¡": "á", "Ã©": "é", "Ã­": "í", "Ã³": "ó",
    "Ãº": "ú", "â": "a", "Ã‰": "É", "Ã“": "Ó",
    "Ãš": "Ú", "Ã±": "ñ", "Ã‘": "Ñ", "í±": "ñ",
    "í§": "ç", "Ã§": "ç", "Ã¼": "ü", "í¼": "ü",
    "â€œ": "“", "â€": "”", "â€˜": "‘", "â€™": "’",
    "â€“": "–", "â€”": "—", "â€¦": "…",
    "Â¿": "¿", "Â¡": "¡", "Ã¿": "¿",
    "Ã¢ÂÂ": "'", "Ã¢ÂÂ,": ",", "Ã¢ÂÂ.": ".",
    "Ã¢ÂÂ¦": "…", "Ã¢ÂÂ ": " ", "ÃÂ": "-"
}

# Corregir caracteres dañados y reemplazar valores vacíos
def fix_mojibake(s):
    if pd.isna(s):
        return ""

    for bad, good in replacements.items():
        s = s.replace(bad, good)
    return s

data["CONTENIDO A ANALIZAR"] = [
    fix_mojibake(t)
    for t in data["CONTENIDO A ANALIZAR"].values
]

# Convertir la intensidad a números para clasificar los comentarios
data["INTENSIDAD"] = pd.to_numeric(
    data["INTENSIDAD"], errors="coerce"
)

# Guardar los textos procesados y sus valores de intensidad
doc = []
value = []

for i in range(0, lines_number):

    # Analizar lingüísticamente el comentario con spaCy
    tmp_doc = nlp(data["CONTENIDO A ANALIZAR"][i])
    doc.append(tmp_doc)

    # Obtener la etiqueta numérica del comentario
    tmp_value = data["INTENSIDAD"][i]
    value.append(tmp_value)

    # Contar palabras excluyendo espacios y signos de puntuación
    numero_palabras = sum(
        1 for token in tmp_doc
        if not token.is_space and not token.is_punct
    )
    total_palabras += numero_palabras

    # Clasificar el comentario según su intensidad
    if tmp_value > 0:
        total_odio += 1
        palabras_odio += numero_palabras
        print(f"Comentario {i+1}: Odio")

    elif tmp_value == 0:
        total_no_odio += 1
        palabras_no_odio += numero_palabras
        print(f"Comentario {i+1}: No odio")

    else:
        total_sin_clasificar += 1
        print(f"Comentario {i+1}: Sin clasificación")

    # Comprobar si spaCy detectó entidades nombradas (NER)
    tiene_ner = len(tmp_doc.ents) > 0

    # IMPORTANTE: no volver a incrementar total_odio ni
    # total_no_odio aquí, porque ya se incrementaron arriba.
    if tmp_value > 0:
        if tiene_ner:
            odio_con_ner += 1

    elif tmp_value == 0:
        if tiene_ner:
            no_odio_con_ner += 1

    # Contar signos de exclamación del comentario
    numero_exclamaciones = tmp_doc.text.count("!")

    if tmp_value > 0:
        exclamaciones_odio += numero_exclamaciones
    elif tmp_value == 0:
        exclamaciones_no_odio += numero_exclamaciones

    # Calcular riqueza léxica: lemas únicos / palabras alfabéticas
    palabras_unicas = set(
        token.lemma_.lower()
        for token in tmp_doc
        if token.is_alpha
    )

    total_palabras_lexicas = sum(
        1 for token in tmp_doc if token.is_alpha
    )

    riqueza_lexica = (
        len(palabras_unicas) / total_palabras_lexicas
        if total_palabras_lexicas > 0 else 0
    )

    # Acumular la riqueza léxica según la clasificación
    if tmp_value > 0:
        riqueza_lexica_odio += riqueza_lexica
    elif tmp_value == 0:
        riqueza_lexica_no_odio += riqueza_lexica

    # Contar pronombres identificados por spaCy
    numero_pronombres = sum(
        1 for token in tmp_doc if token.pos_ == "PRON"
    )

    # Acumular pronombres y palabras de cada grupo
    if tmp_value > 0:
        pronombres_odio += numero_pronombres
        palabras_pronombres_odio += numero_palabras
    elif tmp_value == 0:
        pronombres_no_odio += numero_pronombres
        palabras_pronombres_no_odio += numero_palabras

    # Buscar coincidencias exactas con el diccionario de odio
    terminos_en_comentario = [
        token.text.lower()
        for token in tmp_doc
        if token.is_alpha
        and token.text.lower() in palabras_diccionario_odio
    ]

    numero_terminos_odio = len(terminos_en_comentario)

    # Contar cuántas veces aparece cada término
    for termino in terminos_en_comentario:
        frecuencia_por_termino[termino] = (
            frecuencia_por_termino.get(termino, 0) + 1
        )

    # Separar las frecuencias según la intensidad del comentario
    if tmp_value > 0:
        frecuencia_terminos_odio += numero_terminos_odio
        if numero_terminos_odio > 0:
            comentarios_con_terminos_odio += 1

    elif tmp_value == 0:
        frecuencia_terminos_no_odio += numero_terminos_odio
        if numero_terminos_odio > 0:
            comentarios_con_terminos_no_odio += 1

# Calcular promedios generales y evitar divisiones por cero
promedio_palabras = (
    total_palabras / lines_number if lines_number > 0 else 0
)

porcentaje_odio_ner = (
    odio_con_ner / total_odio * 100 if total_odio > 0 else 0
)

porcentaje_no_odio_ner = (
    no_odio_con_ner / total_no_odio * 100
    if total_no_odio > 0 else 0
)

promedio_odio = (
    palabras_odio / total_odio if total_odio > 0 else 0
)

promedio_no_odio = (
    palabras_no_odio / total_no_odio if total_no_odio > 0 else 0
)

# Promedio de exclamaciones por comentario
promedio_exclamaciones_odio = (
    exclamaciones_odio / total_odio if total_odio > 0 else 0
)

promedio_exclamaciones_no_odio = (
    exclamaciones_no_odio / total_no_odio
    if total_no_odio > 0 else 0
)

# Promedio de riqueza léxica por comentario
promedio_riqueza_odio = (
    riqueza_lexica_odio / total_odio if total_odio > 0 else 0
)

promedio_riqueza_no_odio = (
    riqueza_lexica_no_odio / total_no_odio
    if total_no_odio > 0 else 0
)

# Normalizar la frecuencia de pronombres por cada 100 palabras
promedio_pronombres_odio = (
    pronombres_odio / palabras_pronombres_odio * 100
    if palabras_pronombres_odio > 0 else 0
)

promedio_pronombres_no_odio = (
    pronombres_no_odio / palabras_pronombres_no_odio * 100
    if palabras_pronombres_no_odio > 0 else 0
)

# Calcular el promedio de términos del diccionario por comentario
promedio_terminos_odio = (
    frecuencia_terminos_odio / total_odio if total_odio > 0 else 0
)

promedio_terminos_no_odio = (
    frecuencia_terminos_no_odio / total_no_odio
    if total_no_odio > 0 else 0
)


Comentario 1: Odio
Comentario 2: No odio
Comentario 3: Odio
Comentario 4: Odio
Comentario 5: Odio
Comentario 6: Odio
Comentario 7: Odio
Comentario 8: Odio
Comentario 9: Odio
Comentario 10: Odio
Comentario 11: Odio
Comentario 12: Odio
Comentario 13: No odio
Comentario 14: Odio
Comentario 15: Odio
Comentario 16: Odio
Comentario 17: No odio
Comentario 18: Odio
Comentario 19: No odio
Comentario 20: No odio
Comentario 21: Odio
Comentario 22: Odio
Comentario 23: Odio
Comentario 24: Odio
Comentario 25: Odio
Comentario 26: Odio
Comentario 27: Odio
Comentario 28: Odio
Comentario 29: Odio
Comentario 30: Odio
Comentario 31: Odio
Comentario 32: Odio
Comentario 33: Odio
Comentario 34: No odio
Comentario 35: Odio
Comentario 36: Odio
Comentario 37: Odio
Comentario 38: Odio
Comentario 39: Odio
Comentario 40: Odio
Comentario 41: Odio
Comentario 42: Odio
Comentario 43: No odio
Comentario 44: Odio
Comentario 45: Odio
Comentario 46: Odio
Comentario 47: Odio
Comentario 48: Odio
Comentario 49: Odio
Comentar

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Pregunta 1.</span>
<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">¿Cuántos registros contiene el corpus?</span>

In [118]:
print(len(data))

575029


Implementar len para identificar en data todos los registros del corpus, en la columna "CONTENIDO A ANALIZAR" cuantos registros existen en la columna

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Pregunta 2.</span>
<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">¿Cuántas palabras totales hay en los comentarios del corpus?</span>

In [119]:
total_corpus = data["CONTENIDO A ANALIZAR"].dropna().astype(str).str.split().str.len().sum()
print("Total de corpus:", total_corpus)

Total de corpus: 62999969


analiza cada palabra y la cuenta hasta obtener Total de corpus
 

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Pregunta 3.</span>
<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">¿Cuál el número promedio de palabras en cada comentario?</span>

In [120]:
print(round(promedio_palabras, 2))

21.61


<b>Calcular promedios generales del total y diviendolo por el numero de lineas para obtener el promedio de la muestra de 10000</b>
<hr>
 

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Pregunta 4.</span>
<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Considerando dos grupos de comentarios (odio y no odio) ¿Cuál el número promedio de palabras en los comentarios de cada grupo?</span>

In [121]:
print("\n--- Comentarios con odio ---")
print("Cantidad:", total_odio)
print("Promedio de palabras:", round(promedio_odio, 2))


--- Comentarios con odio ---
Cantidad: 1496
Promedio de palabras: 11.42


<b>Calcular promedios generales del total y diviendolo por el numero de lineas para obtener el promedio de la muestra de 10000</b>
<hr>
 

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Pregunta 5.</span>
<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Considerando dos grupos de comentarios (odio y no odio) ¿Cuál es el número promedio de oraciones en los comentarios de cada grupo?</span>

In [122]:
print("\n--- Comentarios sin odio ---")
print("Cantidad:", total_no_odio)
print("Promedio de palabras:", round(promedio_no_odio, 2))


--- Comentarios sin odio ---
Cantidad: 18504
Promedio de palabras: 10.76


<b>Calcular promedios generales del total y diviendolo por el numero de lineas para obtener el promedio de la muestra de 10000</b>
<hr>
 

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Pregunta 6.</span>
<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Considerando dos grupos de comentarios (odio y no odio) ¿Cuál es el porcentaje de comentarios que contienen entidades NER en cada grupo?</span>

In [123]:
print("Porcentaje con NER no odio: ", round(porcentaje_no_odio_ner, 2), "%")
print("Porcentaje con NER odio: ", round(porcentaje_odio_ner, 2), "%")

Porcentaje con NER no odio:  19.85 %
Porcentaje con NER odio:  22.19 %


<b>Incluye aquí, debajo de la línea, la explicación de tu respuesta</b>
<hr>
 

<span style="font-size: 16pt; font-weight: bold; color: #0098cd;">Plantea tus propias preguntas</span>

<span><b>Plantea al menos 4 características</b> del texto cuyo análisis permita una caracterización completa del texto. Puedes utilizar recomendaciones proporcionadas por la IA Generativa, si así lo deseas. Para cada una de las características planteadas, obtén valores separados para los grupos ODIO/NO-ODIO.</span>

<span>En la explicación aportada, deberás <b>explicar el significado de la característica planteada</b> así como la importancia de ésta en la caracterización del texto.</span>

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Característica adicional 1.</span>


<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">[INDICA AQUÍ LA CARACTERÍSTICA QUE PROPONES]</span>

In [124]:
print("\n--- Signos de exclamación ---")
print(
    "Total de exclamaciones en comentarios con odio:",
    exclamaciones_odio
)
print(
    "Promedio de exclamaciones por comentario con odio:",
    round(promedio_exclamaciones_odio, 2)
)

print(
    "Total de exclamaciones en comentarios sin odio:",
    exclamaciones_no_odio
)
print(
    "Promedio de exclamaciones por comentario sin odio:",
    round(promedio_exclamaciones_no_odio, 2)
)


--- Signos de exclamación ---
Total de exclamaciones en comentarios con odio: 270
Promedio de exclamaciones por comentario con odio: 0.18
Total de exclamaciones en comentarios sin odio: 2287
Promedio de exclamaciones por comentario sin odio: 0.12


<b>Incluye aquí, debajo de la línea, la explicación de la característica propuesta y su motivación. Incluye también una explicación del código fuente aportado.</b>
<hr>
 

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Característica adicional 2.</span>


<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">[INDICA AQUÍ LA CARACTERÍSTICA QUE PROPONES]</span>

In [125]:
print("\n--- Riqueza léxica ---")

print(
    "Promedio de riqueza léxica con odio:",
    round(promedio_riqueza_odio, 3)
)

print(
    "Promedio de riqueza léxica sin odio:",
    round(promedio_riqueza_no_odio, 3)
)


--- Riqueza léxica ---
Promedio de riqueza léxica con odio: 0.453
Promedio de riqueza léxica sin odio: 0.448


<b>Incluye aquí, debajo de la línea, la explicación de la característica propuesta y su motivación. Incluye también una explicación del código fuente aportado.</b>
<hr>
 

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Característica adicional 3.</span>


<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">[INDICA AQUÍ LA CARACTERÍSTICA QUE PROPONES]</span>

In [126]:
print("\n--- Frecuencia de pronombres ---")

print("Total de pronombres en comentarios con odio:",
      pronombres_odio)

print("Pronombres por cada 100 palabras (ODIO):",
      round(promedio_pronombres_odio, 2))

print("Total de pronombres en comentarios sin odio:",
      pronombres_no_odio)

print("Pronombres por cada 100 palabras (NO-ODIO):",
      round(promedio_pronombres_no_odio, 2))


--- Frecuencia de pronombres ---
Total de pronombres en comentarios con odio: 1258
Pronombres por cada 100 palabras (ODIO): 7.36
Total de pronombres en comentarios sin odio: 16040
Pronombres por cada 100 palabras (NO-ODIO): 8.06


<b>Incluye aquí, debajo de la línea, la explicación de la característica propuesta y su motivación. Incluye también una explicación del código fuente aportado.</b>
<hr>
 

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Característica adicional 4.</span>


<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">[INDICA AQUÍ LA CARACTERÍSTICA QUE PROPONES]</span>

In [127]:
print("\n--- FRECUENCIA DE TÉRMINOS DE ODIO ---")

print(f"Frecuencia en comentarios con odio: {frecuencia_terminos_odio}")
print(f"Frecuencia en comentarios sin odio: {frecuencia_terminos_no_odio}")

print(f"Comentarios con odio que contienen términos: {comentarios_con_terminos_odio}")
print(f"Comentarios sin odio que contienen términos: {comentarios_con_terminos_no_odio}")

print(f"Promedio de términos por comentario con odio: {promedio_terminos_odio:.2f}")
print(f"Promedio de términos por comentario sin odio: {promedio_terminos_no_odio:.2f}")

print("\n--- FRECUENCIA POR PALABRA ---")

for termino, frecuencia in sorted(
    frecuencia_por_termino.items(),
    key=lambda item: item[1],
    reverse=True
):
    print(f"{termino}: {frecuencia}")


--- FRECUENCIA DE TÉRMINOS DE ODIO ---
Frecuencia en comentarios con odio: 64
Frecuencia en comentarios sin odio: 134
Comentarios con odio que contienen términos: 54
Comentarios sin odio que contienen términos: 131
Promedio de términos por comentario con odio: 0.04
Promedio de términos por comentario sin odio: 0.01

--- FRECUENCIA POR PALABRA ---
asco: 98
basura: 84
idiota: 9
ignorante: 7


<b>Incluye aquí, debajo de la línea, la explicación de la característica propuesta y su motivación. Incluye también una explicación del código fuente aportado.</b>
<hr>
 

<span style="font-size: 16pt; font-weight: bold; color: #0098cd;">Reflexión final</span>

<span>Una de las utilidades de la caracterización de texto es que nos sirve como etapa de <i>feature-extraction</i> (extración de características) de cara a un posterior sistema de clasificación. Es pertinente, por tanto, reflexionar sobre la capacidad discriminatoria de cada una de las características extraídas. </span>

<span> Responde, para ello, a la siguiente pregunta.</span>

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Reflexión final.</span>
<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">¿Es posible utilizar alguna de las características extraídas en las preguntas anteriores para determinar si un mensaje contiene odio? Justifica tu respuesta con el análisis estadístico que consideres necesario.</span>

In [128]:
# Incluye aquí el código generado para poder responder a tu pregunta


<b>Incluye aquí, debajo de la línea, la explicación de tu respuesta</b>
<hr>
 

In [ ]:
Podemos tener la capacidad de, utilizar las caracteristicas anteriores, para identificar el odio o no de una frase o sentencia, estadisticamente
se puede tener un resultado base con sus respectivas desviaciones para identificar con precisión a traves de numeros mensajes de odio, implementado
a futuro, un aprendizaje automatico para agilizar la identificación de odio. 